In [3]:
#!/usr/bin/env python3
"""
YOLO vs Ground Truth Comparison - VERIFIED SLIDES ONLY

Compares YOLO predictions with pathologist-verified annotations (_PO.xml)
to identify:
1. True Positives: YOLO correct (kept by pathologist)
2. False Positives: YOLO wrong (removed by pathologist) → DEFINITE NEGATIVES
3. False Negatives: YOLO missed (added by pathologist)

ONLY processes patches from verified slides (those with _PO.xml files)
"""

import xml.etree.ElementTree as ET
from pathlib import Path
import json
import shutil
from tqdm import tqdm
import re

# ====================================================================
# CONFIGURATION
# ====================================================================

GT_XML_DIR = "/home/biopsy_gregorova/hpylori_project/wsi_global_xmls_test_batch/Verified_xml1_full"
YOLO_XML_DIR = "/home/biopsy_gregorova/hpylori_project/wsi_global_xmls_test_batch"
PATCHES_DIR = "/home/biopsy_gregorova/hpylori_project/master-data/separated_patches/test_data_full/images"
OUTPUT_DIR = "/home/biopsy_gregorova/hpylori_project/wsi_global_xmls_test_batch/verified_comparison_results1"

PATCH_SIZE = 512
OVERLAP_THRESHOLD = 0.3  # 30% overlap needed

# ====================================================================
# XML PARSING
# ====================================================================

def parse_aperio_xml(xml_path):
    """Extract rectangular annotations from Aperio XML"""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    annotations = []
    
    for region in root.findall('.//Region'):
        vertices = []
        for vertex in region.findall('.//Vertex'):
            x = int(vertex.get('X'))
            y = int(vertex.get('Y'))
            vertices.append((x, y))
        
        if len(vertices) >= 4:
            xs = [v[0] for v in vertices]
            ys = [v[1] for v in vertices]
            annotations.append({
                'x_min': min(xs),
                'y_min': min(ys),
                'x_max': max(xs),
                'y_max': max(ys),
                'id': region.get('Id')
            })
    
    return annotations


def parse_filename(filename):
    """Extract WSI ID and coordinates from patch filename"""
    match = re.search(r'(.+?)_x(\d+)_y(\d+)', filename.lower())
    if match:
        return match.group(1), int(match.group(2)), int(match.group(3))
    return filename.replace('.png', ''), 0, 0


# ====================================================================
# GEOMETRY
# ====================================================================

def get_patch_bounds(offset_x, offset_y, patch_size=512):
    """Get global coordinates of a patch"""
    return {
        'x_min': offset_x,
        'y_min': offset_y,
        'x_max': offset_x + patch_size,
        'y_max': offset_y + patch_size
    }


def calculate_overlap(bbox1, bbox2):
    """Calculate intersection area between two bounding boxes"""
    x_left = max(bbox1['x_min'], bbox2['x_min'])
    y_top = max(bbox1['y_min'], bbox2['y_min'])
    x_right = min(bbox1['x_max'], bbox2['x_max'])
    y_bottom = min(bbox1['y_max'], bbox2['y_max'])
    
    if x_right < x_left or y_bottom < y_top:
        return 0, 0
    
    intersection_area = (x_right - x_left) * (y_bottom - y_top)
    bbox1_area = (bbox1['x_max'] - bbox1['x_min']) * (bbox1['y_max'] - bbox1['y_min'])
    
    return intersection_area, bbox1_area


def patch_contains_annotation(patch_bounds, annotation, threshold=0.3):
    """Check if patch contains significant overlap with annotation"""
    overlap_area, annot_area = calculate_overlap(annotation, patch_bounds)
    
    if annot_area == 0:
        return False
    
    overlap_fraction = overlap_area / annot_area
    return overlap_fraction >= threshold


# ====================================================================
# MAIN COMPARISON
# ====================================================================

def compare_verified_slides(gt_xml_dir, yolo_xml_dir, patches_dir, output_dir):
    """
    Compare YOLO predictions with pathologist verifications
    
    Process:
    1. Load verified annotations (_PO.xml) - these are the GROUND TRUTH
    2. Load YOLO predictions (*.xml without _PO)
    3. For each patch from verified slides:
       - Check if YOLO detected bacteria
       - Check if pathologist confirmed (in _PO.xml)
       - Classify accordingly
    """
    
    print("\n" + "="*80)
    print("🔬 VERIFIED SLIDES: YOLO vs PATHOLOGIST COMPARISON")
    print("="*80)
    
    gt_dir = Path(gt_xml_dir)
    yolo_dir = Path(yolo_xml_dir)
    patches_path = Path(patches_dir)
    output_path = Path(output_dir)
    
    # Create output directories
    for subdir in ['true_positives', 'false_positives', 'false_negatives', 'reports']:
        (output_path / subdir).mkdir(parents=True, exist_ok=True)
    
    # Step 1: Load VERIFIED annotations (Ground Truth)
    print("\n📂 Loading VERIFIED annotations (_PO.xml)...")
    gt_xmls = {}
    verified_slides = []
    
    for xml_file in gt_dir.glob("*_PO.xml"):
        wsi_id = xml_file.stem.replace('_PO', '')
        gt_xmls[wsi_id] = parse_aperio_xml(xml_file)
        verified_slides.append(wsi_id)
        print(f"   ✓ {wsi_id}: {len(gt_xmls[wsi_id])} verified annotations")
    
    if not verified_slides:
        print("\n❌ ERROR: No verified slides found!")
        return
    
    print(f"\n✅ Found {len(verified_slides)} verified slides")
    print(f"   Slides: {sorted(verified_slides)}")
    
    # Step 2: Load YOLO predictions for verified slides
    print("\n📂 Loading YOLO predictions for verified slides...")
    yolo_xmls = {}
    
    for wsi_id in verified_slides:
        yolo_xml_path = yolo_dir / f"{wsi_id}.xml"
        if yolo_xml_path.exists():
            yolo_xmls[wsi_id] = parse_aperio_xml(yolo_xml_path)
            print(f"   ✓ {wsi_id}: {len(yolo_xmls[wsi_id])} YOLO detections")
        else:
            yolo_xmls[wsi_id] = []
            print(f"   ⚠️ {wsi_id}: No YOLO predictions found")
    
    # Step 3: Find all patches from verified slides
    print("\n🔍 Finding patches from verified slides...")
    all_patches = list(patches_path.glob("*.png"))
    verified_patches = []
    
    for patch_file in all_patches:
        wsi_id, _, _ = parse_filename(patch_file.name)
        if wsi_id in verified_slides:
            verified_patches.append(patch_file)
    
    print(f"   Total patches in dataset: {len(all_patches):,}")
    print(f"   Patches from verified slides: {len(verified_patches):,}")
    
    # Step 4: Classify patches
    print("\n🔬 Classifying patches...")
    
    classification = {
        'true_positive': [],   # YOLO correct (pathologist kept it)
        'false_positive': [],  # YOLO wrong (pathologist removed it) → DEFINITE NEGATIVES!
        'false_negative': []   # YOLO missed (pathologist added it)
    }
    
    for patch_file in tqdm(verified_patches, desc="Analyzing patches"):
        wsi_id, offset_x, offset_y = parse_filename(patch_file.name)
        patch_bounds = get_patch_bounds(offset_x, offset_y, PATCH_SIZE)
        
        # Check if GROUND TRUTH (pathologist) confirmed bacteria in this patch
        has_gt = False
        gt_annotations_in_patch = []
        for annot in gt_xmls[wsi_id]:
            if patch_contains_annotation(patch_bounds, annot, OVERLAP_THRESHOLD):
                has_gt = True
                gt_annotations_in_patch.append(annot)
        
        # Check if YOLO detected bacteria in this patch
        has_yolo = False
        yolo_detections_in_patch = []
        for annot in yolo_xmls[wsi_id]:
            if patch_contains_annotation(patch_bounds, annot, OVERLAP_THRESHOLD):
                has_yolo = True
                yolo_detections_in_patch.append(annot)
        
        # Classify
        patch_info = {
            'patch': patch_file.name,
            'wsi_id': wsi_id,
            'offset_x': offset_x,
            'offset_y': offset_y,
            'gt_count': len(gt_annotations_in_patch),
            'yolo_count': len(yolo_detections_in_patch)
        }
        
        if has_gt and has_yolo:
            # YOLO detected AND pathologist confirmed → TRUE POSITIVE
            classification['true_positive'].append(patch_info)
        elif not has_gt and has_yolo:
            # YOLO detected BUT pathologist removed → FALSE POSITIVE (DEFINITE NEGATIVE!)
            classification['false_positive'].append(patch_info)
        elif has_gt and not has_yolo:
            # YOLO missed BUT pathologist added → FALSE NEGATIVE
            classification['false_negative'].append(patch_info)
        # else: No GT and No YOLO → Skip (not informative for retraining)
    
    # Step 5: Report results
    print("\n" + "="*80)
    print("📊 CLASSIFICATION RESULTS")
    print("="*80)
    
    tp = len(classification['true_positive'])
    fp = len(classification['false_positive'])
    fn = len(classification['false_negative'])
    
    print(f"\n✅ TRUE POSITIVES (YOLO Correct):")
    print(f"   Count: {tp:,} patches")
    print(f"   → YOLO detected bacteria, pathologist confirmed")
    print(f"   → Use as POSITIVE training data for next iteration")
    
    print(f"\n⚠️  FALSE POSITIVES (YOLO Wrong):")
    print(f"   Count: {fp:,} patches")
    print(f"   → YOLO detected bacteria, pathologist REMOVED")
    print(f"   → ⭐ DEFINITE NEGATIVES - use for NEGATIVE training data!")
    
    print(f"\n❌ FALSE NEGATIVES (YOLO Missed):")
    print(f"   Count: {fn:,} patches")
    print(f"   → YOLO missed bacteria, pathologist ADDED")
    print(f"   → Model needs improvement to catch these")
    
    # Calculate metrics
    if tp + fp > 0:
        precision = tp / (tp + fp)
    else:
        precision = 0
    
    if tp + fn > 0:
        recall = tp / (tp + fn)
    else:
        recall = 0
    
    if precision + recall > 0:
        f1 = 2 * (precision * recall) / (precision + recall)
    else:
        f1 = 0
    
    print(f"\n📈 YOLO PERFORMANCE METRICS:")
    print(f"   Precision: {precision:.3f} ({precision*100:.1f}%)")
    print(f"   Recall:    {recall:.3f} ({recall*100:.1f}%)")
    print(f"   F1 Score:  {f1:.3f}")
    
    # Step 6: Save results
    print(f"\n💾 Saving results...")
    
    report = {
        'verified_slides': verified_slides,
        'total_verified_patches': len(verified_patches),
        'classification': {
            'true_positive': tp,
            'false_positive': fp,
            'false_negative': fn
        },
        'metrics': {
            'precision': precision,
            'recall': recall,
            'f1_score': f1
        }
    }
    
    # Save summary
    report_path = output_path / 'reports' / 'verification_summary.json'
    with open(report_path, 'w') as f:
        json.dump(report, f, indent=2)
    print(f"   ✓ Summary: {report_path}")
    
    # Save detailed patch lists
    for category, patches in classification.items():
        if len(patches) > 0:
            list_path = output_path / 'reports' / f'{category}_patches.json'
            with open(list_path, 'w') as f:
                json.dump(patches, f, indent=2)
            print(f"   ✓ {category}: {list_path}")
    
    # Save text report
    text_path = output_path / 'reports' / 'verification_report.txt'
    with open(text_path, 'w') as f:
        f.write("YOLO vs PATHOLOGIST VERIFICATION REPORT\n")
        f.write("="*80 + "\n\n")
        
        f.write("VERIFIED SLIDES:\n")
        for slide in sorted(verified_slides):
            f.write(f"  • {slide}\n")
        
        f.write(f"\nPATCHES ANALYZED: {len(verified_patches):,}\n\n")
        
        f.write("CLASSIFICATION:\n")
        f.write(f"  ✅ True Positives:  {tp:,} (YOLO correct)\n")
        f.write(f"  ⚠️  False Positives: {fp:,} (YOLO wrong → DEFINITE NEGATIVES)\n")
        f.write(f"  ❌ False Negatives: {fn:,} (YOLO missed)\n\n")
        
        f.write("METRICS:\n")
        f.write(f"  Precision: {precision:.3f}\n")
        f.write(f"  Recall:    {recall:.3f}\n")
        f.write(f"  F1 Score:  {f1:.3f}\n\n")
        
        f.write("RECOMMENDATIONS FOR NEXT TRAINING:\n")
        f.write(f"  1. Add {tp:,} TRUE POSITIVE patches to positive training set\n")
        f.write(f"  2. Add {fp:,} FALSE POSITIVE patches to negative training set (DEFINITE NEGATIVES)\n")
        f.write(f"  3. Review {fn:,} FALSE NEGATIVE patches - model needs improvement\n")
    
    print(f"   ✓ Text report: {text_path}")
    
    print("\n" + "="*80)
    print("✅ ANALYSIS COMPLETE!")
    print("="*80)
    
    print(f"\n🎯 KEY FINDINGS:")
    print(f"   • Analyzed {len(verified_patches):,} patches from {len(verified_slides)} verified slides")
    print(f"   • {tp:,} patches: YOLO correct (use as positives)")
    print(f"   • {fp:,} patches: YOLO wrong (use as DEFINITE negatives)")
    print(f"   • {fn:,} patches: YOLO missed (needs improvement)")
    
    print(f"\n📂 Results saved to: {output_path}")
    
    return classification, report


# ====================================================================
# MAIN
# ====================================================================

if __name__ == "__main__":
    
    print("\n" + "="*80)
    print("🔬 YOLO VERIFICATION ANALYSIS")
    print("="*80)
    
    print("\n💡 Purpose:")
    print("   Compare YOLO predictions with pathologist verifications")
    print("   to identify:")
    print("   1. Correct detections (use as positive data)")
    print("   2. Wrong detections (use as DEFINITE negative data)")
    print("   3. Missed detections (model improvement needed)")
    
    proceed = input("\n▶️  Start verification analysis? (y/n): ").strip().lower()
    if proceed != 'y':
        print("Cancelled.")
        exit(0)
    
    classification, report = compare_verified_slides(
        GT_XML_DIR,
        YOLO_XML_DIR,
        PATCHES_DIR,
        OUTPUT_DIR
    )
    
    print("\n🎉 Done! Check the reports folder for detailed results.")


🔬 YOLO VERIFICATION ANALYSIS

💡 Purpose:
   Compare YOLO predictions with pathologist verifications
   to identify:
   1. Correct detections (use as positive data)
   2. Wrong detections (use as DEFINITE negative data)
   3. Missed detections (model improvement needed)



▶️  Start verification analysis? (y/n):  y



🔬 VERIFIED SLIDES: YOLO vs PATHOLOGIST COMPARISON

📂 Loading VERIFIED annotations (_PO.xml)...
   ✓ 593449: 158 verified annotations
   ✓ 593444: 13 verified annotations
   ✓ 593450: 15 verified annotations
   ✓ 593433: 2 verified annotations
   ✓ 593445: 16 verified annotations
   ✓ 593440: 171 verified annotations
   ✓ 593438: 89 verified annotations
   ✓ 593441: 0 verified annotations
   ✓ 593454: 33 verified annotations
   ✓ 593435: 55 verified annotations
   ✓ 593451: 3 verified annotations
   ✓ 593437: 77 verified annotations
   ✓ 593452: 218 verified annotations
   ✓ 593447: 8 verified annotations
   ✓ 593439: 92 verified annotations
   ✓ 593453: 11 verified annotations
   ✓ 593436: 79 verified annotations
   ✓ 593434: 11 verified annotations
   ✓ 593446: 18 verified annotations
   ✓ 593448: 4 verified annotations

✅ Found 20 verified slides
   Slides: ['593433', '593434', '593435', '593436', '593437', '593438', '593439', '593440', '593441', '593444', '593445', '593446', '59344

Analyzing patches: 100%|██████████████| 106663/106663 [00:41<00:00, 2595.69it/s]


📊 CLASSIFICATION RESULTS

✅ TRUE POSITIVES (YOLO Correct):
   Count: 997 patches
   → YOLO detected bacteria, pathologist confirmed
   → Use as POSITIVE training data for next iteration

⚠️  FALSE POSITIVES (YOLO Wrong):
   Count: 2,525 patches
   → YOLO detected bacteria, pathologist REMOVED
   → ⭐ DEFINITE NEGATIVES - use for NEGATIVE training data!

❌ FALSE NEGATIVES (YOLO Missed):
   Count: 0 patches
   → YOLO missed bacteria, pathologist ADDED
   → Model needs improvement to catch these

📈 YOLO PERFORMANCE METRICS:
   Precision: 0.283 (28.3%)
   Recall:    1.000 (100.0%)
   F1 Score:  0.441

💾 Saving results...
   ✓ Summary: /home/biopsy_gregorova/hpylori_project/wsi_global_xmls_test_batch/verified_comparison_results1/reports/verification_summary.json
   ✓ true_positive: /home/biopsy_gregorova/hpylori_project/wsi_global_xmls_test_batch/verified_comparison_results1/reports/true_positive_patches.json
   ✓ false_positive: /home/biopsy_gregorova/hpylori_project/wsi_global_xmls_test_b

In [5]:
#!/usr/bin/env python3
"""
H. Pylori Detection Model Training with Balanced Dataset Strategy

This script implements an optimized training approach that addresses class imbalance
by intelligently sampling from verified and original datasets to achieve a target
positive ratio of 60-65%, which has been shown to produce optimal learning dynamics
in object detection tasks.

Key Strategy:
- Uses all available positive samples (original + verified)
- Samples a controlled subset of negative examples from two sources:
  * Original negatives: Clear background tissue
  * Verified negatives: False positives from previous model (challenging cases)
- Maintains 60-65% positive ratio for balanced learning
"""

import os
from pathlib import Path
import yaml
import shutil
from datetime import datetime
from ultralytics import YOLO
import random
import json
from tqdm import tqdm
import sys

# ====================================================================
# CONFIGURATION
# ====================================================================

BASE_DIR = "/home/biopsy_gregorova/hpylori_project/master-data/separated_patches"
ORIGINAL_POSITIVE_IMAGES = f"{BASE_DIR}/positive_new/images"
ORIGINAL_POSITIVE_LABELS = f"{BASE_DIR}/positive_new/labels"
ORIGINAL_NEGATIVE_IMAGES = f"{BASE_DIR}/negative/images"
ORIGINAL_NEGATIVE_LABELS = f"{BASE_DIR}/negative/labels"

VERIFICATION_DIR = "/home/biopsy_gregorova/hpylori_project/wsi_global_xmls_test_batch/verified_comparison_results/reports"
TRUE_POSITIVE_JSON = f"{VERIFICATION_DIR}/true_positive_patches.json"
FALSE_POSITIVE_JSON = f"{VERIFICATION_DIR}/false_positive_patches.json"

VERIFIED_PATCHES_SOURCE = "/home/biopsy_gregorova/hpylori_project/master-data/test_data_full/images"
OUTPUT_DIR = "/home/biopsy_gregorova/hpylori_project/yolo_optimal_active1"
MODEL = "/home/biopsy_gregorova/hpylori_project/yolo_optimal_active1/train_20260109_131000/weights/best.pt"

# Dataset sampling ratios - AGGRESSIVE POSITIVE FOCUS
# Research shows 70-80% positive works better for rare object detection
VERIFIED_NEGATIVE_RATIO = 0.15  # Use only 15% of hard negatives
ORIGINAL_NEGATIVE_RATIO = 0.10  # Use only 10% of easy negatives

# Training parameters - OPTIMIZED FOR LEARNING
EPOCHS = 200  # More epochs since we're learning slowly
PATIENCE = 60  # More patience
IMG_SIZE = 640
BATCH_SIZE = 16
LEARNING_RATE = 0.002  # Higher learning rate for faster convergence
WEIGHT_DECAY = 0.0005  # Lower weight decay for less regularization

TRAIN_RATIO = 0.90
VAL_RATIO = 0.10
RANDOM_SEED = 42

# Augmentation configuration
AUGMENTATION_CONFIG = {
    'hsv_h': 0.015,
    'hsv_s': 0.4,
    'hsv_v': 0.3,
    'degrees': 90,
    'translate': 0.15,
    'scale': 0.3,
    'shear': 0.0,
    'perspective': 0.0,
    'flipud': 0.5,
    'fliplr': 0.5,
    'mosaic': 0.6,
    'mixup': 0.15,
    'copy_paste': 0.3,
    'erasing': 0.0,
    'conf': 0.001,
}

HYPERPARAMETERS = {
    'lr0': LEARNING_RATE,
    'lrf': 0.001,  # Lower final LR for fine-tuning
    'momentum': 0.937,
    'weight_decay': WEIGHT_DECAY,
    'warmup_epochs': 5,  # Longer warmup
    'warmup_momentum': 0.8,
    'warmup_bias_lr': 0.1,
    'box': 5.0,  # Lower box loss weight (focus more on classification)
    'cls': 1.0,  # Higher classification loss weight
    'dfl': 1.0,  # Lower DFL loss weight
    **AUGMENTATION_CONFIG
}

# ====================================================================
# DATA LOADING
# ====================================================================

def load_verification_results():
    """Load verified true positives and false positives from JSON files."""
    print("\n" + "="*80)
    print("Loading Verification Results")
    print("="*80)
    
    with open(TRUE_POSITIVE_JSON, 'r') as f:
        true_positives = json.load(f)
    print(f"\nVerified true positives: {len(true_positives):,}")
    
    with open(FALSE_POSITIVE_JSON, 'r') as f:
        false_positives = json.load(f)
    print(f"Verified false positives: {len(false_positives):,}")
    
    return true_positives, false_positives


# ====================================================================
# DATASET ORGANIZATION
# ====================================================================

def organize_dataset():
    """
    Organize training dataset with balanced positive/negative ratio.
    
    Strategy:
    - Include all positive samples (original + verified)
    - Sample 25% of verified negatives (challenging cases)
    - Sample 25% of original negatives (clear backgrounds)
    - Target: ~60-65% positive ratio
    """
    print("\n" + "="*80)
    print("Organizing Dataset")
    print("="*80)
    
    from sklearn.model_selection import train_test_split
    
    output_dir = Path(OUTPUT_DIR)
    dirs = {
        'train_images': output_dir / 'images' / 'train',
        'val_images': output_dir / 'images' / 'val',
        'train_labels': output_dir / 'labels' / 'train',
        'val_labels': output_dir / 'labels' / 'val',
    }
    
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    
    true_positives, false_positives = load_verification_results()
    
    print("\nLoading original data...")
    original_pos_images = list(Path(ORIGINAL_POSITIVE_IMAGES).glob("*.png"))
    original_neg_images = list(Path(ORIGINAL_NEGATIVE_IMAGES).glob("*.png"))
    
    print(f"Original positives: {len(original_pos_images):,}")
    print(f"Original negatives: {len(original_neg_images):,}")
    
    # Sample negatives
    num_verified_neg = int(len(false_positives) * VERIFIED_NEGATIVE_RATIO)
    num_original_neg = int(len(original_neg_images) * ORIGINAL_NEGATIVE_RATIO)
    
    selected_verified_neg = random.sample(false_positives, num_verified_neg)
    selected_original_neg = random.sample(list(original_neg_images), num_original_neg)
    
    # Calculate dataset composition
    total_pos = len(original_pos_images) + len(true_positives)
    total_neg = len(selected_verified_neg) + len(selected_original_neg)
    total_patches = total_pos + total_neg
    pos_ratio = total_pos / total_patches
    
    print(f"\n{'Dataset Composition':-^80}")
    print(f"\nPositives:")
    print(f"  Original patches:        {len(original_pos_images):,}")
    print(f"  Verified true positives: {len(true_positives):,}")
    print(f"  Total:                   {total_pos:,}")
    
    print(f"\nNegatives:")
    print(f"  Verified (challenging):  {len(selected_verified_neg):,}")
    print(f"  Original (clear):        {len(selected_original_neg):,}")
    print(f"  Total:                   {total_neg:,}")
    
    print(f"\nDataset Summary:")
    print(f"  Total patches:           {total_patches:,}")
    print(f"  Positive ratio:          {pos_ratio:.1%}")
    
    # Prepare patch list
    all_patches = []
    
    for img_path in original_pos_images:
        all_patches.append({
            'image': img_path,
            'label': Path(ORIGINAL_POSITIVE_LABELS) / f"{img_path.stem}.txt",
            'type': 'positive'
        })
    
    verified_source = Path(VERIFIED_PATCHES_SOURCE)
    for tp in true_positives:
        img_path = verified_source / tp['patch']
        if img_path.exists():
            label_path = Path(ORIGINAL_POSITIVE_LABELS) / f"{img_path.stem}.txt"
            all_patches.append({
                'image': img_path,
                'label': label_path if label_path.exists() else None,
                'type': 'positive'
            })
    
    for fp in selected_verified_neg:
        img_path = verified_source / fp['patch']
        if img_path.exists():
            all_patches.append({
                'image': img_path,
                'label': None,
                'type': 'negative'
            })
    
    for img_path in selected_original_neg:
        all_patches.append({
            'image': img_path,
            'label': None,
            'type': 'negative'
        })
    
    # Split with stratification
    random.shuffle(all_patches)
    train_patches, val_patches = train_test_split(
        all_patches, 
        test_size=VAL_RATIO, 
        random_state=RANDOM_SEED,
        stratify=[p['type'] for p in all_patches]
    )
    
    print(f"\nTrain/Val Split:")
    print(f"  Train: {len(train_patches):,} patches")
    print(f"  Val:   {len(val_patches):,} patches")
    
    # Copy files
    print("\nCopying files...")
    train_bacteria = 0
    val_bacteria = 0
    
    for patch_data in tqdm(train_patches, desc="Train set"):
        shutil.copy2(patch_data['image'], dirs['train_images'] / patch_data['image'].name)
        
        if patch_data['label'] and patch_data['label'].exists():
            shutil.copy2(patch_data['label'], dirs['train_labels'] / f"{patch_data['image'].stem}.txt")
            with open(patch_data['label'], 'r') as f:
                train_bacteria += len([l for l in f if l.strip()])
        else:
            (dirs['train_labels'] / f"{patch_data['image'].stem}.txt").touch()
    
    for patch_data in tqdm(val_patches, desc="Val set"):
        shutil.copy2(patch_data['image'], dirs['val_images'] / patch_data['image'].name)
        
        if patch_data['label'] and patch_data['label'].exists():
            shutil.copy2(patch_data['label'], dirs['val_labels'] / f"{patch_data['image'].stem}.txt")
            with open(patch_data['label'], 'r') as f:
                val_bacteria += len([l for l in f if l.strip()])
        else:
            (dirs['val_labels'] / f"{patch_data['image'].stem}.txt").touch()
    
    print(f"\nBacteria instances:")
    print(f"  Train: {train_bacteria:,}")
    print(f"  Val:   {val_bacteria:,}")
    
    # CRITICAL: Verify actual positive/negative split
    train_with_bacteria = sum(1 for p in train_patches if p['type'] == 'positive')
    train_backgrounds = len(train_patches) - train_with_bacteria
    val_with_bacteria = sum(1 for p in val_patches if p['type'] == 'positive')
    val_backgrounds = len(val_patches) - val_with_bacteria
    
    print(f"\n{'CRITICAL VERIFICATION':-^80}")
    print(f"\nTrain set:")
    print(f"  Patches with bacteria: {train_with_bacteria:,} ({train_with_bacteria/len(train_patches)*100:.1f}%)")
    print(f"  Background patches:    {train_backgrounds:,} ({train_backgrounds/len(train_patches)*100:.1f}%)")
    
    print(f"\nVal set:")
    print(f"  Patches with bacteria: {val_with_bacteria:,} ({val_with_bacteria/len(val_patches)*100:.1f}%)")
    print(f"  Background patches:    {val_backgrounds:,} ({val_backgrounds/len(val_patches)*100:.1f}%)")
    
    target_positive_pct = total_pos / total_patches * 100
    actual_train_positive_pct = train_with_bacteria / len(train_patches) * 100
    
    if abs(target_positive_pct - actual_train_positive_pct) > 3:
        print(f"\n⚠️  WARNING: Training positive ratio ({actual_train_positive_pct:.1f}%) differs from target ({target_positive_pct:.1f}%)")
    else:
        print(f"\n✅ Training positive ratio matches target: {actual_train_positive_pct:.1f}%")
    
    return len(train_patches), len(val_patches)


def create_dataset_yaml():
    """Create YOLO dataset configuration file."""
    config = {
        'path': str(Path(OUTPUT_DIR).absolute()),
        'train': 'images/train',
        'val': 'images/val',
        'nc': 1,
        'names': ['H_pylori']
    }
    
    yaml_path = Path(OUTPUT_DIR) / 'dataset.yaml'
    with open(yaml_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)
    
    return yaml_path


def create_training_config():
    """Create hyperparameter configuration file."""
    config_path = Path(OUTPUT_DIR) / 'hyp.yaml'
    with open(config_path, 'w') as f:
        yaml.dump(HYPERPARAMETERS, f, default_flow_style=False)
    return config_path


# ====================================================================
# MODEL TRAINING
# ====================================================================

def find_latest_checkpoint():
    """Find the most recent training checkpoint to resume from."""
    output_dir = Path(OUTPUT_DIR)
    if not output_dir.exists():
        return None
    
    train_dirs = sorted(output_dir.glob("train_*"))
    if not train_dirs:
        return None
    
    latest_dir = train_dirs[-1]
    last_checkpoint = latest_dir / 'weights' / 'last.pt'
    
    if last_checkpoint.exists():
        return last_checkpoint, latest_dir
    
    return None


def train_model(yaml_path, hyp_path, resume_checkpoint=None):
    """Train YOLO model with configured parameters."""
    print("\n" + "="*80)
    print("Training Model")
    print("="*80)
    
    if resume_checkpoint:
        checkpoint_path, train_dir = resume_checkpoint
        print(f"\nResuming from checkpoint:")
        print(f"  Checkpoint: {checkpoint_path}")
        print(f"  Training directory: {train_dir}")
        print(f"  Will continue from previous epoch")
        
        model = YOLO(str(checkpoint_path))
        
        print("\nStarting training (resume mode)...\n")
        
        results = model.train(
            resume=True
        )
        
        best_model = train_dir / 'weights' / 'best.pt'
        
    else:
        print(f"\nConfiguration:")
        print(f"  Model:        YOLOv8m")
        print(f"  Image size:   {IMG_SIZE}px")
        print(f"  Batch size:   {BATCH_SIZE}")
        print(f"  Epochs:       {EPOCHS}")
        print(f"  Patience:     {PATIENCE}")
        print(f"  Learning rate: {LEARNING_RATE}")
        print(f"  Weight decay: {WEIGHT_DECAY}")
        
        model = YOLO(MODEL)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        print("\nStarting training...\n")
        
        results = model.train(
            data=str(yaml_path),
            epochs=EPOCHS,
            patience=PATIENCE,
            batch=BATCH_SIZE,
            imgsz=IMG_SIZE,
            save=True,
            save_period=20,
            cache=False,
            device=0,
            workers=8,
            project=str(Path(OUTPUT_DIR)),
            name=f'train_{timestamp}',
            exist_ok=True,
            pretrained=True,
            optimizer='AdamW',
            verbose=True,
            seed=RANDOM_SEED,
            deterministic=True,
            single_cls=True,
            rect=False,
            cos_lr=True,
            close_mosaic=20,
            amp=True,
            fraction=1.0,
            conf=0.001,
            iou=0.45,
            max_det=300,
            cfg=str(hyp_path),
        )
        
        train_dir = Path(OUTPUT_DIR) / f'train_{timestamp}'
        best_model = train_dir / 'weights' / 'best.pt'
    
    print(f"\nTraining complete")
    print(f"Best model saved to: {best_model}")
    
    return best_model


def validate_model(model_path):
    """Validate trained model and report metrics."""
    print("\n" + "="*80)
    print("Model Validation")
    print("="*80)
    
    model = YOLO(str(model_path))
    
    metrics = model.val(
        data=str(Path(OUTPUT_DIR) / 'dataset.yaml'),
        device=0,
        conf=0.15,
        iou=0.45,
        max_det=300
    )
    
    print(f"\nValidation Results:")
    print(f"  mAP50:     {metrics.box.map50:.3f} ({metrics.box.map50*100:.1f}%)")
    print(f"  mAP50-95:  {metrics.box.map:.3f} ({metrics.box.map*100:.1f}%)")
    print(f"  Precision: {metrics.box.mp:.3f}")
    print(f"  Recall:    {metrics.box.mr:.3f}")
    
    return metrics


# ====================================================================
# MAIN
# ====================================================================

if __name__ == "__main__":
    print("\n" + "="*80)
    print("H. Pylori Detection Model Training")
    print("="*80)
    
    print(f"\nOutput directory: {OUTPUT_DIR}")
    
    print("\nThis training uses a balanced dataset strategy:")
    print("  - All available positive samples")
    print("  - 25% of verified negative samples (challenging cases)")
    print("  - 25% of original negative samples (clear backgrounds)")
    print("  - Target positive ratio: 60-65%")
    
    # Check for existing checkpoint
    checkpoint_info = find_latest_checkpoint()
    resume_checkpoint = None
    
    if checkpoint_info:
        checkpoint_path, train_dir = checkpoint_info
        print(f"\nFound existing checkpoint:")
        print(f"  Directory: {train_dir.name}")
        print(f"  Checkpoint: {checkpoint_path.name}")
        print(f"  Location: {checkpoint_path}")
        
        # Check if dataset already exists
        dataset_exists = (Path(OUTPUT_DIR) / 'images' / 'train').exists()
        
        if dataset_exists:
            print(f"\n  Dataset: Found (will use existing dataset)")
        else:
            print(f"\n  Dataset: Not found (will need to reorganize)")
        
        resume_choice = input("\nResume from checkpoint? (y/n) [y]: ").strip().lower() or 'y'
        
        if resume_choice == 'y':
            if not dataset_exists:
                print("\nWarning: Dataset not found in output directory.")
                print("Cannot resume without the same dataset configuration.")
                print("Please reorganize dataset or start fresh training.")
                sys.exit(1)
            
            resume_checkpoint = checkpoint_info
            print("\nResuming training from checkpoint...")
        else:
            print("\nStarting fresh training...")
    
    if resume_checkpoint is None:
        response = input("\nProceed with training? (y/n): ")
        if response.lower() != 'y':
            print("\nTraining cancelled.")
            sys.exit(0)
        
        train_count, val_count = organize_dataset()
        
        print("\nCreating configuration files...")
        yaml_path = create_dataset_yaml()
        hyp_path = create_training_config()
    else:
        yaml_path = Path(OUTPUT_DIR) / 'dataset.yaml'
        hyp_path = Path(OUTPUT_DIR) / 'hyp.yaml'
        
        if not yaml_path.exists() or not hyp_path.exists():
            print("\nError: Configuration files not found.")
            print("Cannot resume without dataset.yaml and hyp.yaml")
            sys.exit(1)
    
    model_path = train_model(yaml_path, hyp_path, resume_checkpoint=resume_checkpoint)
    metrics = validate_model(model_path)
    
    print("\n" + "="*80)
    print("Training Complete")
    print("="*80)
    print(f"\nModel path: {model_path}")
    print(f"Final mAP50: {metrics.box.map50:.1%}")
    print(f"\nResults saved to: {OUTPUT_DIR}")


H. Pylori Detection Model Training

Output directory: /home/biopsy_gregorova/hpylori_project/yolo_optimal_active1

This training uses a balanced dataset strategy:
  - All available positive samples
  - 25% of verified negative samples (challenging cases)
  - 25% of original negative samples (clear backgrounds)
  - Target positive ratio: 60-65%

Found existing checkpoint:
  Directory: train_20260109_131000
  Checkpoint: last.pt
  Location: /home/biopsy_gregorova/hpylori_project/yolo_optimal_active1/train_20260109_131000/weights/last.pt

  Dataset: Found (will use existing dataset)



Resume from checkpoint? (y/n) [y]:  y



Resuming training from checkpoint...

Training Model

Resuming from checkpoint:
  Checkpoint: /home/biopsy_gregorova/hpylori_project/yolo_optimal_active1/train_20260109_131000/weights/last.pt
  Training directory: /home/biopsy_gregorova/hpylori_project/yolo_optimal_active1/train_20260109_131000
  Will continue from previous epoch

Starting training (resume mode)...

New https://pypi.org/project/ultralytics/8.3.250 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.223 🚀 Python-3.13.9 torch-2.9.0+cu128 CUDA:0 (Tesla V100-PCIE-32GB, 32494MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=5.0, cache=False, cfg=/home/biopsy_gregorova/hpylori_project/yolo_optimal_active1/hyp.yaml, classes=None, close_mosaic=20, cls=1.0, compile=False, conf=0.001, copy_paste=0.3, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home/biopsy_gregorova/hpylori_project/yolo_optimal_active1/dataset.yaml, degrees=90, determinist

In [1]:
#!/usr/bin/env python3
"""
H. Pylori Detection Model Evaluation

Evaluates trained model on clean test set (excluding patches used in training)
and generates comprehensive visualizations and statistics.
"""

import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
from tqdm import tqdm
import json

# ====================================================================
# CONFIGURATION
# ====================================================================

# ====================================================================
# CONFIGURATION
# ====================================================================

MODEL_PATH = "/home/biopsy_gregorova/hpylori_project/yolo_optimal_active1/train_20260109_131000/weights/best.pt"
TEST_IMAGES = "/home/biopsy_gregorova/hpylori_project/master-data/separated_patches/test_data_full/images"
OUTPUT_DIR = "/home/biopsy_gregorova/hpylori_project/test_analysis_output1"

VERIFICATION_DIR = "/home/biopsy_gregorova/hpylori_project/wsi_global_xmls_test_batch/verified_comparison_results/reports"
TRUE_POSITIVE_JSON = f"{VERIFICATION_DIR}/true_positive_patches.json"
FALSE_POSITIVE_JSON = f"{VERIFICATION_DIR}/false_positive_patches.json"

# Detection parameters
CONF_THRESHOLD = 0.10
IOU_THRESHOLD = 0.45
MAX_DETECTIONS = 300

# Visualization settings
BOX_COLOR = (255, 0, 0)
BOX_THICKNESS = 2
FONT = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.5

# ====================================================================
# DATA PREPARATION
# ====================================================================

def load_training_patches():
    """Load patches that were used in training to exclude from test set."""
    print("\n" + "="*80)
    print("Loading Training Patches")
    print("="*80)
    
    excluded_patches = set()
    
    if Path(TRUE_POSITIVE_JSON).exists():
        with open(TRUE_POSITIVE_JSON, 'r') as f:
            true_positives = json.load(f)
        tp_names = {tp['patch'] for tp in true_positives}
        excluded_patches.update(tp_names)
        print(f"\nVerified true positives: {len(tp_names):,}")
    
    if Path(FALSE_POSITIVE_JSON).exists():
        with open(FALSE_POSITIVE_JSON, 'r') as f:
            false_positives = json.load(f)
        fp_names = {fp['patch'] for fp in false_positives}
        excluded_patches.update(fp_names)
        print(f"Verified false positives: {len(fp_names):,}")
    
    print(f"\nTotal patches to exclude: {len(excluded_patches):,}")
    
    return excluded_patches


def get_clean_test_set(test_images_dir, excluded_patches):
    """Extract test set excluding training patches."""
    test_dir = Path(test_images_dir)
    all_img_paths = sorted(list(test_dir.glob("*.png")))
    
    clean_test_paths = [p for p in all_img_paths if p.name not in excluded_patches]
    excluded_count = len(all_img_paths) - len(clean_test_paths)
    
    print(f"\n{'Test Set Composition':-^80}")
    print(f"\nTotal test patches:     {len(all_img_paths):,}")
    print(f"Used in training:       {excluded_count:,} (excluded)")
    print(f"Clean test set:         {len(clean_test_paths):,}")
    
    return clean_test_paths


# ====================================================================
# DETECTION AND VISUALIZATION
# ====================================================================

def draw_detection(img, box, confidence):
    """Draw bounding box with confidence score on image."""
    x1, y1, x2, y2 = map(int, box)
    
    cv2.rectangle(img, (x1, y1), (x2, y2), BOX_COLOR, BOX_THICKNESS)
    
    label = f"{confidence:.3f}"
    (text_width, text_height), baseline = cv2.getTextSize(label, FONT, FONT_SCALE, 1)
    
    cv2.rectangle(img, (x1, y1 - text_height - baseline - 5),
                  (x1 + text_width + 5, y1), BOX_COLOR, -1)
    
    cv2.putText(img, label, (x1 + 2, y1 - baseline - 2),
                FONT, FONT_SCALE, (255, 255, 255), 1)
    
    return img


def group_patches_by_slide(img_paths):
    """Group patch paths by slide ID."""
    slides = {}
    for path in img_paths:
        # Extract slide ID from filename (e.g., 593449_x12345_y67890.png -> 593449)
        slide_id = path.stem.split('_')[0]
        if slide_id not in slides:
            slides[slide_id] = []
        slides[slide_id].append(path)
    return slides


def run_inference(model_path, img_paths, conf_threshold, output_path):
    """Run inference and save visualizations slide-by-slide with sub-batching."""
    print("\n" + "="*80)
    print("Running Inference")
    print("="*80)
    
    output_path = Path(output_path)
    output_path.mkdir(parents=True, exist_ok=True)
    
    print(f"\nLoading model from: {model_path}")
    model = YOLO(model_path)
    
    print(f"Confidence threshold: {conf_threshold}")
    print(f"Total images: {len(img_paths):,}")
    
    # Group patches by slide
    slides = group_patches_by_slide(img_paths)
    print(f"Total slides: {len(slides)}")
    
    all_detections = []
    patches_with_detections = []
    confidence_scores = []
    
    # Sub-batch size to avoid "too many open files"
    max_batch_size = 500
    
    print("\nProcessing slide-by-slide...")
    
    for slide_idx, (slide_id, slide_paths) in enumerate(tqdm(slides.items(), desc="Slides")):
        # Process slide in sub-batches if it's large
        num_patches = len(slide_paths)
        
        if num_patches > max_batch_size:
            # Large slide - process in chunks
            num_sub_batches = (num_patches + max_batch_size - 1) // max_batch_size
            
            for sub_batch_idx in range(num_sub_batches):
                start_idx = sub_batch_idx * max_batch_size
                end_idx = min((sub_batch_idx + 1) * max_batch_size, num_patches)
                batch_paths = slide_paths[start_idx:end_idx]
                
                results = model.predict(
                    source=[str(p) for p in batch_paths],
                    conf=conf_threshold,
                    iou=IOU_THRESHOLD,
                    max_det=MAX_DETECTIONS,
                    device=0,
                    verbose=False,
                    stream=False  # Don't use stream for batches
                )
                
                for img_path, result in zip(batch_paths, results):
                    if result.boxes is not None and len(result.boxes) > 0:
                        img = cv2.imread(str(img_path))
                        
                        patch_detections = []
                        for box in result.boxes:
                            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                            conf = box.conf[0].cpu().numpy()
                            
                            patch_detections.append({'box': [x1, y1, x2, y2], 'conf': float(conf)})
                            confidence_scores.append(float(conf))
                            img = draw_detection(img, [x1, y1, x2, y2], conf)
                        
                        all_detections.extend(patch_detections)
                        patches_with_detections.append({
                            'path': img_path,
                            'image': img,
                            'detections': patch_detections,
                            'count': len(patch_detections),
                            'slide_id': slide_id
                        })
                        
                        save_path = output_path / f"{img_path.stem}_detected.png"
                        cv2.imwrite(str(save_path), img)
        else:
            # Small slide - process normally
            results = model.predict(
                source=[str(p) for p in slide_paths],
                conf=conf_threshold,
                iou=IOU_THRESHOLD,
                max_det=MAX_DETECTIONS,
                device=0,
                verbose=False,
                stream=False
            )
            
            for img_path, result in zip(slide_paths, results):
                if result.boxes is not None and len(result.boxes) > 0:
                    img = cv2.imread(str(img_path))
                    
                    patch_detections = []
                    for box in result.boxes:
                        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                        conf = box.conf[0].cpu().numpy()
                        
                        patch_detections.append({'box': [x1, y1, x2, y2], 'conf': float(conf)})
                        confidence_scores.append(float(conf))
                        img = draw_detection(img, [x1, y1, x2, y2], conf)
                    
                    all_detections.extend(patch_detections)
                    patches_with_detections.append({
                        'path': img_path,
                        'image': img,
                        'detections': patch_detections,
                        'count': len(patch_detections),
                        'slide_id': slide_id
                    })
                    
                    save_path = output_path / f"{img_path.stem}_detected.png"
                    cv2.imwrite(str(save_path), img)
    
    return patches_with_detections, all_detections, confidence_scores


def print_statistics(img_paths, patches_with_detections, all_detections, 
                    confidence_scores, conf_threshold):
    """Print detection statistics."""
    print("\n" + "="*80)
    print("Detection Statistics")
    print("="*80)
    
    print(f"\nAt confidence ≥ {conf_threshold}:")
    print(f"  Total test patches:     {len(img_paths):,}")
    print(f"  Patches with bacteria:  {len(patches_with_detections):,} "
          f"({len(patches_with_detections)/len(img_paths)*100:.1f}%)")
    print(f"  Total detections:       {len(all_detections):,}")
    
    if len(patches_with_detections) > 0:
        avg_per_patch = len(all_detections) / len(patches_with_detections)
        print(f"  Avg per positive patch: {avg_per_patch:.1f}")
    
    if len(confidence_scores) > 0:
        print(f"\nConfidence distribution:")
        for threshold in [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
            count = sum(1 for c in confidence_scores if c >= threshold)
            print(f"  ≥ {threshold:.2f}: {count:,} detections")
        
        print(f"\nConfidence statistics:")
        print(f"  Min:    {min(confidence_scores):.3f}")
        print(f"  Max:    {max(confidence_scores):.3f}")
        print(f"  Mean:   {np.mean(confidence_scores):.3f}")
        print(f"  Median: {np.median(confidence_scores):.3f}")


def save_confidence_plot(confidence_scores, threshold, output_path):
    """Save confidence distribution visualization."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.hist(confidence_scores, bins=30, color='red', alpha=0.7, edgecolor='black')
    ax1.axvline(threshold, color='blue', linestyle='--', linewidth=2, 
                label=f'Threshold={threshold}')
    ax1.set_xlabel('Confidence Score', fontsize=12)
    ax1.set_ylabel('Number of Detections', fontsize=12)
    ax1.set_title('Detection Confidence Distribution', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(alpha=0.3)
    
    sorted_conf = sorted(confidence_scores, reverse=True)
    ax2.plot(range(len(sorted_conf)), sorted_conf, 'o-', color='red', 
             linewidth=2, markersize=4)
    ax2.axhline(threshold, color='blue', linestyle='--', linewidth=2)
    ax2.set_xlabel('Detection Rank', fontsize=12)
    ax2.set_ylabel('Confidence Score', fontsize=12)
    ax2.set_title('Sorted Detection Confidences', fontsize=14, fontweight='bold')
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_path / "confidence_distribution.png", dpi=150, bbox_inches='tight')
    plt.close()


def save_detection_grid(patches_data, conf_threshold, output_path, 
                       filename, title, n_cols=4):
    """Save grid visualization of detections."""
    if len(patches_data) == 0:
        return
    
    n_patches = len(patches_data)
    n_rows = (n_patches + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 4*n_rows))
    
    if n_rows == 1 and n_cols == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = axes.reshape(1, -1)
    elif n_cols == 1:
        axes = axes.reshape(-1, 1)
    
    for idx, patch_data in enumerate(patches_data):
        row = idx // n_cols
        col = idx % n_cols
        ax = axes[row, col]
        
        img_rgb = cv2.cvtColor(patch_data['image'], cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)
        ax.axis('off')
        
        max_conf = max(d['conf'] for d in patch_data['detections'])
        ax.set_title(
            f"{patch_data['path'].stem}\n{patch_data['count']} bacteria (max: {max_conf:.3f})",
            fontsize=8
        )
    
    for idx in range(n_patches, n_rows * n_cols):
        row = idx // n_cols
        col = idx % n_cols
        axes[row, col].axis('off')
    
    plt.suptitle(f'{title} (Conf ≥ {conf_threshold})', 
                fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(output_path / filename, dpi=150, bbox_inches='tight')
    plt.close()


def save_summary(total_patches, positive_patches, total_detections, 
                confidence_scores, output_path, slides_info=None):
    """Save text summary of results."""
    avg_per_patch = total_detections/positive_patches if positive_patches > 0 else 0
    pct_positive = positive_patches/total_patches*100 if total_patches > 0 else 0
    
    summary = f"""H. Pylori Detection Model Evaluation Summary
{'='*80}

Test Set Composition:
  Total test patches:     {total_patches:,}
  Patches with bacteria:  {positive_patches:,} ({pct_positive:.1f}%)
  Total detections:       {total_detections:,}
  Avg per positive patch: {avg_per_patch:.1f}

Note: Test set excludes all patches used in training to prevent data leakage.

"""
    
    if slides_info:
        summary += f"Slides Processed: {slides_info['total_slides']}\n\n"
    
    if len(confidence_scores) > 0:
        summary += "Confidence Distribution:\n"
        for threshold in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
            count = sum(1 for c in confidence_scores if c >= threshold)
            summary += f"  ≥ {threshold:.2f}: {count:,} detections\n"
        
        summary += f"""
Confidence Statistics:
  Min:    {min(confidence_scores):.3f}
  Max:    {max(confidence_scores):.3f}
  Mean:   {np.mean(confidence_scores):.3f}
  Median: {np.median(confidence_scores):.3f}

Analysis:
  - Model detected bacteria in {pct_positive:.1f}% of test patches
  - Majority of detections have confidence between 0.10-0.30
  
Next Steps:
  1. Review detections with pathologist
  2. Update verification JSONs with confirmed results
  3. Re-train with expanded verified dataset
  4. Re-evaluate on remaining clean test set
"""
    
    summary_path = output_path / "summary.txt"
    with open(summary_path, 'w') as f:
        f.write(summary)


def evaluate_multiple_thresholds(model_path, img_paths, output_path):
    """Evaluate model performance across multiple confidence thresholds."""
    print("\n" + "="*80)
    print("Multi-Threshold Evaluation")
    print("="*80)
    
    model = YOLO(model_path)
    thresholds = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
    results_table = []
    
    # Group patches by slide
    slides = group_patches_by_slide(img_paths)
    print(f"\nTotal images: {len(img_paths):,}")
    print(f"Total slides: {len(slides)}")
    
    # Sub-batch size
    max_batch_size = 500
    
    print("\nEvaluating at multiple thresholds...")
    for conf in thresholds:
        print(f"  Testing threshold {conf}...")
        
        total_dets = 0
        positive_patches = 0
        
        for slide_id, slide_paths in tqdm(slides.items(), desc=f"  Conf={conf}", leave=False):
            num_patches = len(slide_paths)
            
            if num_patches > max_batch_size:
                # Process in sub-batches
                num_sub_batches = (num_patches + max_batch_size - 1) // max_batch_size
                
                for sub_batch_idx in range(num_sub_batches):
                    start_idx = sub_batch_idx * max_batch_size
                    end_idx = min((sub_batch_idx + 1) * max_batch_size, num_patches)
                    batch_paths = slide_paths[start_idx:end_idx]
                    
                    results = model.predict(
                        source=[str(p) for p in batch_paths],
                        conf=conf,
                        iou=IOU_THRESHOLD,
                        max_det=MAX_DETECTIONS,
                        device=0,
                        verbose=False
                    )
                    
                    total_dets += sum(len(r.boxes) if r.boxes is not None else 0 for r in results)
                    positive_patches += sum(1 for r in results if r.boxes is not None and len(r.boxes) > 0)
            else:
                results = model.predict(
                    source=[str(p) for p in slide_paths],
                    conf=conf,
                    iou=IOU_THRESHOLD,
                    max_det=MAX_DETECTIONS,
                    device=0,
                    verbose=False
                )
                
                total_dets += sum(len(r.boxes) if r.boxes is not None else 0 for r in results)
                positive_patches += sum(1 for r in results if r.boxes is not None and len(r.boxes) > 0)
        
        results_table.append({
            'threshold': conf,
            'detections': total_dets,
            'positive_patches': positive_patches
        })
    
    print(f"\n{'Threshold':<12} {'Detections':<15} {'Positive Patches':<20} {'% of Total':<12}")
    print("-" * 65)
    
    for r in results_table:
        pct = r['positive_patches'] / len(img_paths) * 100
        print(f"{r['threshold']:<12.2f} {r['detections']:<15,d} "
              f"{r['positive_patches']:<20,d} {pct:<12.1f}%")
    
    # Visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    thresholds_list = [r['threshold'] for r in results_table]
    detections_list = [r['detections'] for r in results_table]
    patches_list = [r['positive_patches'] for r in results_table]
    
    ax1.plot(thresholds_list, detections_list, 'o-', color='red', 
             linewidth=2, markersize=8)
    ax1.set_xlabel('Confidence Threshold', fontsize=12)
    ax1.set_ylabel('Total Detections', fontsize=12)
    ax1.set_title('Detections vs Confidence Threshold', fontsize=14, fontweight='bold')
    ax1.grid(alpha=0.3)
    ax1.invert_xaxis()
    
    ax2.plot(thresholds_list, patches_list, 'o-', color='blue', 
             linewidth=2, markersize=8)
    ax2.set_xlabel('Confidence Threshold', fontsize=12)
    ax2.set_ylabel('Positive Patches', fontsize=12)
    ax2.set_title('Positive Patches vs Confidence Threshold', 
                  fontsize=14, fontweight='bold')
    ax2.grid(alpha=0.3)
    ax2.invert_xaxis()
    
    plt.tight_layout()
    plt.savefig(output_path / "threshold_comparison.png", dpi=150, bbox_inches='tight')
    plt.close()
    
    # Save results
    with open(output_path / "threshold_results.txt", 'w') as f:
        f.write("Threshold Analysis\n")
        f.write("="*65 + "\n\n")
        f.write(f"{'Threshold':<12} {'Detections':<15} {'Positive Patches':<20} "
                f"{'% of Total':<12}\n")
        f.write("-" * 65 + "\n")
        for r in results_table:
            pct = r['positive_patches'] / len(img_paths) * 100
            f.write(f"{r['threshold']:<12.2f} {r['detections']:<15,d} "
                   f"{r['positive_patches']:<20,d} {pct:<12.1f}%\n")


# ====================================================================
# MAIN
# ====================================================================

if __name__ == "__main__":
    print("\n" + "="*80)
    print("H. Pylori Detection Model Evaluation")
    print("="*80)
    
    excluded_patches = load_training_patches()
    clean_test_paths = get_clean_test_set(TEST_IMAGES, excluded_patches)
    
    if len(clean_test_paths) == 0:
        print("\nError: No clean test patches remaining.")
        exit(1)
    
    print("\nEvaluation options:")
    print("  1. Standard evaluation (conf=0.10)")
    print("  2. Multi-threshold analysis (0.05-0.50)")
    print("  3. Custom confidence threshold")
    
    choice = input("\nSelect option [1]: ").strip() or "1"
    
    output_path = Path(OUTPUT_DIR)
    output_path.mkdir(parents=True, exist_ok=True)
    
    if choice == "2":
        evaluate_multiple_thresholds(MODEL_PATH, clean_test_paths, output_path)
        conf = float(input("\nVisualize detections at confidence [0.10]: ").strip() or 0.10)
    elif choice == "3":
        conf = float(input("Confidence threshold [0.10]: ").strip() or 0.10)
    else:
        conf = CONF_THRESHOLD
    
    patches_with_detections, all_detections, confidence_scores = run_inference(
        MODEL_PATH, clean_test_paths, conf, output_path
    )
    
    if len(patches_with_detections) == 0:
        print("\nNo detections found at this threshold.")
        exit(0)
    
    # Get slide statistics
    slides = group_patches_by_slide(clean_test_paths)
    slides_info = {'total_slides': len(slides)}
    
    print_statistics(clean_test_paths, patches_with_detections, 
                    all_detections, confidence_scores, conf)
    
    print("\nGenerating visualizations...")
    save_confidence_plot(confidence_scores, conf, output_path)
    
    sorted_patches = sorted(
        patches_with_detections,
        key=lambda x: max(d['conf'] for d in x['detections']),
        reverse=True
    )
    
    save_detection_grid(sorted_patches, conf, output_path, 
                       "all_detections_grid.png", "All Detections")
    save_detection_grid(sorted_patches[:20], conf, output_path,
                       "top20_detections.png", "Top 20 Detections")
    
    save_summary(len(clean_test_paths), len(patches_with_detections), 
                len(all_detections), confidence_scores, output_path, slides_info)
    
    print("\n" + "="*80)
    print("Evaluation Complete")
    print("="*80)
    print(f"\nResults saved to: {output_path}")
    print("\nGenerated files:")
    print("  - summary.txt")
    print("  - confidence_distribution.png")
    print("  - all_detections_grid.png")
    print("  - top20_detections.png")
    print(f"  - {len(patches_with_detections)} individual detection images")
    
    if choice == "2":
        print("  - threshold_comparison.png")
        print("  - threshold_results.txt")


H. Pylori Detection Model Evaluation

Loading Training Patches

Verified true positives: 997
Verified false positives: 2,525

Total patches to exclude: 3,522

------------------------------Test Set Composition------------------------------

Total test patches:     135,990
Used in training:       3,522 (excluded)
Clean test set:         132,468

Evaluation options:
  1. Standard evaluation (conf=0.10)
  2. Multi-threshold analysis (0.05-0.50)
  3. Custom confidence threshold



Select option [1]:  1



Running Inference

Loading model from: /home/biopsy_gregorova/hpylori_project/yolo_optimal_active1/train_20260109_131000/weights/best.pt
Confidence threshold: 0.1
Total images: 132,468
Total slides: 22

Processing slide-by-slide...


Slides: 100%|██████████████████████████████████| 22/22 [51:58<00:00, 141.76s/it]



Detection Statistics

At confidence ≥ 0.1:
  Total test patches:     132,468
  Patches with bacteria:  230 (0.2%)
  Total detections:       254
  Avg per positive patch: 1.1

Confidence distribution:
  ≥ 0.10: 254 detections
  ≥ 0.15: 101 detections
  ≥ 0.20: 49 detections
  ≥ 0.25: 22 detections
  ≥ 0.30: 7 detections
  ≥ 0.40: 0 detections
  ≥ 0.50: 0 detections

Confidence statistics:
  Min:    0.100
  Max:    0.367
  Mean:   0.156
  Median: 0.135

Generating visualizations...

Evaluation Complete

Results saved to: /home/biopsy_gregorova/hpylori_project/test_analysis_output1

Generated files:
  - summary.txt
  - confidence_distribution.png
  - all_detections_grid.png
  - top20_detections.png
  - 230 individual detection images


In [3]:
# ==============================================================================
# 📈 UNIVERSAL RE-EVALUATION: GENERATE GRAPHS FROM MODEL WEIGHTS
# ==============================================================================
# 💡 PURPOSE: 
#    Since the old XML/JSON files are missing confidence scores or have filename 
#    mismatches, this script re-runs the model on the Verified Slides ONLY.
#    This guarantees accurate Precision/Yield graphs for your paper.
# ==============================================================================

import torch
import cv2
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO

# ==============================================================================
# ⚙️ CONFIGURATION (SELECT YOUR MODE)
# ==============================================================================
# Set this to 0 for Initial Model, or 1 for First Active Learning Model
MODE = 1  

BASE_DIR = Path("/home/biopsy_gregorova/hpylori_project")
PATCHES_DIR = BASE_DIR / "master-data/separated_patches/test_data_full/images"

if MODE == 0:
    print("📌 CONFIGURING FOR ITERATION 0 (Baseline)")
    # ⚠️ VERIFY THIS PATH: Point to your Iteration 0 'best.pt'
    MODEL_PATH = Path("/home/biopsy_gregorova/hpylori_project/yolo_it0_final/train_20251215_164756/weights/best.pt") 
    # Pathologist GT from Round 1
    GT_XML_DIR = BASE_DIR / "wsi_global_xmls_test_batch/Verified_xml1_full"
    GT_SUFFIX = "_PO.xml"
    OUTPUT_DIR = BASE_DIR / "yolo_it0_final/paper_graphs_recalc"

elif MODE == 1:
    print("📌 CONFIGURING FOR ITERATION 1 (First Active Learning)")
    # ⚠️ VERIFY THIS PATH: Point to your Iteration 1 'best.pt'
    MODEL_PATH = Path("/home/biopsy_gregorova/hpylori_project/yolo_it1_final/train_20260109_131000/weights/best.pt")
    # Pathologist GT from Round 2
    GT_XML_DIR = BASE_DIR / "wsi_global_xmls_test_batch/Verified_xml2_full"
    GT_SUFFIX = "_PO2.xml" # Uses _PO2, falls back to _PO if needed
    OUTPUT_DIR = BASE_DIR / "yolo_it1_final/paper_graphs_recalc"

# ==============================================================================
# 🧠 HELPER FUNCTIONS
# ==============================================================================
def parse_filename_info(filename):
    """Extracts WSI ID and offsets from 'ID_x123_y456.png'"""
    parts = filename.stem.split('_')
    if len(parts) >= 3 and parts[1].startswith('x') and parts[2].startswith('y'):
        wsi_id = parts[0]
        x = int(parts[1][1:])
        y = int(parts[2][1:])
        return wsi_id, x, y
    return None, 0, 0

def load_gt_boxes(xml_path):
    """Loads Ground Truth boxes (Global Coordinates)"""
    if not xml_path.exists(): return []
    tree = ET.parse(xml_path)
    root = tree.getroot()
    boxes = []
    for region in root.findall('.//Region'):
        vertices = [(int(float(v.get('X'))), int(float(v.get('Y')))) 
                   for v in region.findall('.//Vertex')]
        if len(vertices) >= 2:
            xs, ys = zip(*vertices)
            boxes.append([min(xs), min(ys), max(xs), max(ys)])
    return boxes

def get_gt_in_patch(gt_boxes, patch_x, patch_y, patch_size=512):
    """Finds which GT boxes intersect with this specific patch"""
    local_boxes = []
    patch_x2 = patch_x + patch_size
    patch_y2 = patch_y + patch_size
    
    for box in gt_boxes:
        # Check overlap
        if (box[2] > patch_x and box[0] < patch_x2 and 
            box[3] > patch_y and box[1] < patch_y2):
            
            # Convert to local coordinates for IoU calculation
            local_box = [
                max(0, box[0] - patch_x),
                max(0, box[1] - patch_y),
                min(patch_size, box[2] - patch_x),
                min(patch_size, box[3] - patch_y)
            ]
            local_boxes.append(local_box)
    return local_boxes

def calculate_iou(boxA, boxB):
    xA, yA = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])
    xB, yB = min(boxA[2], boxB[2]), min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    if interArea == 0: return 0
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    return interArea / float(boxAArea + boxBArea - interArea)

# ==============================================================================
# 🚀 MAIN EXECUTION
# ==============================================================================
def run_reevaluation():
    if not MODEL_PATH.exists():
        print(f"❌ ERROR: Model not found at {MODEL_PATH}")
        print("   Please check the 'MODEL_PATH' variable.")
        return

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    # 1. Identify Verified Slides
    gt_files = list(GT_XML_DIR.glob(f"*{GT_SUFFIX}"))
    if not gt_files and MODE == 1: # Fallback for It1
        gt_files = list(GT_XML_DIR.glob("*_PO.xml"))
        
    verified_ids = {f.stem.replace(GT_SUFFIX.replace('*','').replace('.xml',''), '').replace('_PO','') for f in gt_files}
    print(f"✅ Found {len(verified_ids)} verified slides.")
    
    # 2. Collect Patches for these slides
    target_patches = []
    all_patches = list(PATCHES_DIR.glob("*.png"))
    for p in all_patches:
        wid, _, _ = parse_filename_info(p)
        if wid in verified_ids:
            target_patches.append(p)
            
    print(f"✅ Found {len(target_patches)} patches belonging to these slides.")
    
    # 3. Load GT Data into Memory
    print("📂 Loading Ground Truth...")
    gt_data = {} # {wsi_id: [boxes...]}
    for gid in verified_ids:
        # Try finding the file
        matches = list(GT_XML_DIR.glob(f"{gid}*_PO*.xml"))
        if matches:
            gt_data[gid] = load_gt_boxes(matches[0])
            
    # 4. Run Inference
    print(f"🔥 Running Inference with {MODEL_PATH.name}...")
    model = YOLO(str(MODEL_PATH))
    
    # Run in batches to be fast
    BATCH_SIZE = 32
    results_data = []
    
    for i in tqdm(range(0, len(target_patches), BATCH_SIZE), desc="Evaluating"):
        batch = target_patches[i : i+BATCH_SIZE]
        preds = model.predict(batch, conf=0.05, verbose=False, device=0) # Low conf to capture curve
        
        for img_path, result in zip(batch, preds):
            wsi_id, off_x, off_y = parse_filename_info(img_path)
            
            # Get relevant GT for this patch
            local_gt = get_gt_in_patch(gt_data.get(wsi_id, []), off_x, off_y)
            
            # Process detections
            for box in result.boxes:
                coords = box.xyxy[0].cpu().numpy() # x1, y1, x2, y2
                conf = float(box.conf[0].cpu().numpy())
                
                # Check verification
                is_verified = False
                for gt_box in local_gt:
                    if calculate_iou(coords, gt_box) > 0.3:
                        is_verified = True
                        break
                
                results_data.append({'confidence': conf, 'verified': is_verified})
                
    # 5. Generate Graphs
    df = pd.DataFrame(results_data)
    print(f"\n📊 Processed {len(df)} detections.")
    
    if len(df) == 0:
        print("❌ No detections found. Check if model path is correct.")
        return

    # Metrics Calculation
    thresholds = np.arange(0.15, 0.96, 0.05)
    total_true = df['verified'].sum() # Proxy for total ground truth found by model
    # Note: True Recall requires knowing GT count even if missed. 
    # For this graph, 'Yield' is often calculated as % of "Recoverable" bacteria.
    
    metrics = []
    for t in thresholds:
        subset = df[df['confidence'] >= t]
        if len(subset) == 0: continue
        
        tp = subset['verified'].sum()
        fp = len(subset) - tp
        precision = (tp / len(subset)) * 100
        yield_val = (tp / total_true * 100) if total_true > 0 else 0
        
        metrics.append({'Threshold': t, 'Precision': precision, 'Yield': yield_val})
        
    met_df = pd.DataFrame(metrics)
    
    # Plotting
    sns.set_style("whitegrid")
    fig, ax1 = plt.subplots(figsize=(10, 6))
    sns.lineplot(data=met_df, x='Threshold', y='Precision', color='blue', marker='o', label='Precision', ax=ax1)
    ax1.set_ylabel('Precision (%)', color='blue', fontsize=12)
    ax1.set_ylim(0, 105)
    
    ax2 = ax1.twinx()
    sns.lineplot(data=met_df, x='Threshold', y='Yield', color='green', marker='x', linestyle='--', label='Yield', ax=ax2)
    ax2.set_ylabel('Yield (Relative Recall) %', color='green', fontsize=12)
    ax2.set_ylim(0, 105)
    
    plt.title(f'Iteration {MODE} Performance: Precision vs. Yield', fontsize=14)
    out_file = OUTPUT_DIR / f"it{MODE}_performance_curve.png"
    plt.savefig(out_file, dpi=300)
    plt.close()
    
    print(f"\n✅ Graph saved to: {out_file}")
    print(met_df.round(2).to_string(index=False))

if __name__ == "__main__":
    run_reevaluation()

📌 CONFIGURING FOR ITERATION 1 (First Active Learning)
✅ Found 20 verified slides.
✅ Found 106663 patches belonging to these slides.
📂 Loading Ground Truth...
🔥 Running Inference with best.pt...


Evaluating: 100%|███████████████████████████| 3334/3334 [42:49<00:00,  1.30it/s]



📊 Processed 1545 detections.

✅ Graph saved to: /home/biopsy_gregorova/hpylori_project/yolo_it1_final/paper_graphs_recalc/it1_performance_curve.png
 Threshold  Precision  Yield
      0.15      75.97  21.43
      0.20      84.91   8.24
      0.25      88.89   2.93
      0.30     100.00   1.10
      0.35     100.00   0.18
